# 🚀 Mininet PCAP Training & Prediction - Unified Notebook

**Complete End-to-End Pipeline:**
1. Upload ONE combined PCAP file (normal + attacks)
2. Intelligent attack detection & labeling
3. Train ML models (RF, XGBoost, Ensemble)
4. Comprehensive evaluation & visualizations
5. Make predictions on NEW PCAP files
6. Download models & results

**For Google Colab** - No manual configuration needed!

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q scapy pandas numpy scikit-learn xgboost imbalanced-learn matplotlib seaborn
print("✓ All dependencies installed successfully")

## 📚 Step 2: Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import defaultdict
import joblib

# Colab file handling
from google.colab import files

# Scapy for PCAP parsing
from scapy.all import rdpcap, IP, TCP, UDP, ICMP

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)

import xgboost as xgb
from imblearn.over_sampling import SMOTE

print("✓ All libraries imported")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 📤 Step 3: Upload Training PCAP File

Upload your **combined PCAP file** containing both normal and attack traffic.

In [ ]:
print("="*60)
print("UPLOAD TRAINING PCAP FILE")
print("="*60)
print("\nUpload ONE PCAP file containing:")
print("  - Normal network traffic")
print("  - Attack traffic (SYN flood, port scan, etc.)")
print("\nClick 'Choose Files' below...\n")

uploaded = files.upload()
pcap_file = list(uploaded.keys())[0]

print(f"\n✓ Uploaded: {pcap_file}")
print(f"  Size: {len(uploaded[pcap_file]):,} bytes ({len(uploaded[pcap_file])/1024/1024:.1f} MB)")

## 🔍 Step 4: Intelligent Feature Extraction

This extracts network features and automatically labels traffic as normal or attack.

In [ ]:
class IntelligentPCAPExtractor:
    """Extract features and intelligently label traffic"""
    
    def detect_attack_type(self, features):
        """Detect if flow is attack based on characteristics"""
        
        # SYN Flood: High SYN ratio + high packet rate
        if features['syn_ratio'] > 0.8 and features['packets_per_sec'] > 100:
            return 1, 'syn_flood'
        
        # Port Scan: Few packets + high RST ratio
        if features['packet_count'] < 5 and features['rst_ratio'] > 0.5:
            return 1, 'port_scan'
        
        # UDP Flood: UDP protocol + high packet rate
        if features['protocol'] == 'UDP' and features['packets_per_sec'] > 50:
            return 1, 'udp_flood'
        
        # HTTP Flood: Port 80/443 + high packet rate
        if features['dst_port'] in [80, 443] and features['packets_per_sec'] > 50:
            return 1, 'http_flood'
        
        # Suspicious high packet rate
        if features['packets_per_sec'] > 200:
            return 1, 'ddos'
        
        # Normal traffic
        return 0, 'normal'
    
    def extract_from_pcap(self, pcap_file):
        """Extract and label features from PCAP"""
        print(f"\nProcessing: {pcap_file}")
        
        try:
            packets = rdpcap(pcap_file)
            print(f"  Total packets: {len(packets):,}")
        except Exception as e:
            print(f"  ❌ Error reading PCAP: {e}")
            return []
        
        # Group packets by flow
        flows = defaultdict(list)
        for pkt in packets:
            if IP in pkt:
                flow_key = self._get_flow_key(pkt)
                if flow_key:
                    flows[flow_key].append(pkt)
        
        print(f"  Flows identified: {len(flows):,}")
        
        # Extract features for each flow
        features_list = []
        for flow_key, flow_packets in flows.items():
            feature = self._extract_flow_features(flow_key, flow_packets)
            if feature:
                # Intelligent labeling
                label, attack_type = self.detect_attack_type(feature)
                feature['label'] = label
                feature['attack_type'] = attack_type
                features_list.append(feature)
        
        print(f"  ✓ Extracted {len(features_list):,} flows")
        
        # Show distribution
        normal = sum(1 for f in features_list if f['label'] == 0)
        attack = sum(1 for f in features_list if f['label'] == 1)
        print(f"\n  Distribution:")
        print(f"    Normal: {normal:,} ({normal/len(features_list)*100:.1f}%)")
        print(f"    Attack: {attack:,} ({attack/len(features_list)*100:.1f}%)")
        
        return features_list
    
    def _get_flow_key(self, pkt):
        if IP not in pkt:
            return None
        
        src_ip, dst_ip = pkt[IP].src, pkt[IP].dst
        
        if TCP in pkt:
            return (src_ip, dst_ip, pkt[TCP].sport, pkt[TCP].dport, 'TCP')
        elif UDP in pkt:
            return (src_ip, dst_ip, pkt[UDP].sport, pkt[UDP].dport, 'UDP')
        elif ICMP in pkt:
            return (src_ip, dst_ip, 0, 0, 'ICMP')
        return None
    
    def _extract_flow_features(self, flow_key, packets):
        src_ip, dst_ip, src_port, dst_port, protocol = flow_key
        
        if len(packets) == 0:
            return None
        
        # Timing features
        timestamps = [float(pkt.time) for pkt in packets]
        duration = max(timestamps) - min(timestamps) if len(timestamps) > 1 else 0.001
        
        # Size features
        packet_sizes = [len(pkt) for pkt in packets]
        packet_count = len(packets)
        byte_count = sum(packet_sizes)
        
        # TCP flags
        syn_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x02)
        fin_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x01)
        rst_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x04)
        psh_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x08)
        ack_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x10)
        
        # Derived features
        packets_per_sec = packet_count / duration
        bytes_per_sec = byte_count / duration
        mean_packet_size = np.mean(packet_sizes)
        std_packet_size = np.std(packet_sizes) if len(packet_sizes) > 1 else 0
        
        # Inter-arrival times
        if len(timestamps) > 1:
            iat = np.diff(timestamps)
            mean_iat, std_iat = np.mean(iat), np.std(iat)
        else:
            mean_iat, std_iat = 0, 0
        
        return {
            'duration': duration,
            'protocol': protocol,
            'src_port': src_port,
            'dst_port': dst_port,
            'packet_count': packet_count,
            'byte_count': byte_count,
            'packets_per_sec': packets_per_sec,
            'bytes_per_sec': bytes_per_sec,
            'mean_packet_size': mean_packet_size,
            'std_packet_size': std_packet_size,
            'min_packet_size': min(packet_sizes),
            'max_packet_size': max(packet_sizes),
            'mean_inter_arrival_time': mean_iat,
            'std_inter_arrival_time': std_iat,
            'syn_count': syn_count,
            'fin_count': fin_count,
            'rst_count': rst_count,
            'psh_count': psh_count,
            'ack_count': ack_count,
            'syn_ratio': syn_count / packet_count if packet_count > 0 else 0,
            'fin_ratio': fin_count / packet_count if packet_count > 0 else 0,
            'rst_ratio': rst_count / packet_count if packet_count > 0 else 0,
            'psh_ratio': psh_count / packet_count if packet_count > 0 else 0,
            'ack_ratio': ack_count / packet_count if packet_count > 0 else 0,
            'is_well_known_port': 1 if dst_port < 1024 else 0
        }

print("✓ Feature extractor defined")

## 🔄 Step 5: Process PCAP File

In [ ]:
print("="*60)
print("PROCESSING PCAP FILE")
print("="*60)

extractor = IntelligentPCAPExtractor()
features = extractor.extract_from_pcap(pcap_file)

# Convert to DataFrame
df = pd.DataFrame(features)

print(f"\n{'='*60}")
print(f"✓ EXTRACTION COMPLETE")
print(f"{'='*60}")
print(f"Total samples: {len(df):,}")
print(f"Features: {len(df.columns)}")
print(f"\nClass Distribution:")
print(f"  Normal: {len(df[df['label'] == 0]):,}")
print(f"  Attack: {len(df[df['label'] == 1]):,}")

# Verify both classes present
if len(df['label'].unique()) < 2:
    print("\n⚠️ WARNING: Only one class detected!")
    print("The PCAP may contain only normal or only attack traffic.")
    print("Training will proceed but results may be poor.")
else:
    print("\n✓ Both classes present - ready for training!")

print("="*60)

# Display sample
df.head(10)

## 📊 Step 6: Data Analysis & Visualization

In [ ]:
# Attack type distribution
print("\nAttack Type Distribution:")
print(df['attack_type'].value_counts())

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Class distribution
ax = axes[0, 0]
df['label'].value_counts().plot(kind='bar', ax=ax, color=['green', 'red'])
ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.set_xticklabels(['Normal', 'Attack'], rotation=0)
ax.set_ylabel('Count')

# Attack types
ax = axes[0, 1]
df['attack_type'].value_counts().plot(kind='bar', ax=ax, color='coral')
ax.set_title('Attack Type Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Type')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)

# Packets per second
ax = axes[1, 0]
df[df['label'] == 0]['packets_per_sec'].hist(bins=50, ax=ax, alpha=0.7, label='Normal', color='green')
df[df['label'] == 1]['packets_per_sec'].hist(bins=50, ax=ax, alpha=0.7, label='Attack', color='red')
ax.set_title('Packets Per Second Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Packets/sec')
ax.legend()
ax.set_xlim(0, df['packets_per_sec'].quantile(0.95))

# Protocol distribution
ax = axes[1, 1]
df['protocol'].value_counts().plot(kind='pie', ax=ax, autopct='%1.1f%%')
ax.set_title('Protocol Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig('data_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Visualizations saved")

## 🔧 Step 7: Data Preprocessing

In [ ]:
print("\nPreprocessing data...")

# Separate features and labels
X = df.drop(['label', 'attack_type'], axis=1)
y = df['label']

# Encode categorical features
label_encoders = {}
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le
    print(f"  Encoded: {col}")

# Handle missing/infinite values
X = X.fillna(0).replace([np.inf, -np.inf], 0)

print(f"\n✓ Preprocessing complete")
print(f"  Features: {len(X.columns)}")
print(f"  Samples: {len(X):,}")

## ✂️ Step 8: Train/Val/Test Split & Feature Engineering

In [ ]:
# Split data (60/20/20)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Data Split:")
print(f"  Train: {len(X_train):,} samples")
print(f"  Val:   {len(X_val):,} samples")
print(f"  Test:  {len(X_test):,} samples")

# Scale features
print(f"\nScaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print("✓ Features scaled")

# Feature selection
print(f"\nSelecting top features...")
k_features = min(25, X_train.shape[1])
selector = SelectKBest(mutual_info_classif, k=k_features)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_val_selected = selector.transform(X_val_scaled)
X_test_selected = selector.transform(X_test_scaled)

selected_features = X.columns[selector.get_support()].tolist()
print(f"✓ Selected {len(selected_features)} features")

# Show top features
feature_scores = pd.DataFrame({
    'feature': X.columns,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print(f"\nTop 10 Features:")
for i, row in feature_scores.head(10).iterrows():
    print(f"  {row['feature']}: {row['score']:.4f}")

# SMOTE balancing (only if both classes present)
if len(np.unique(y_train)) > 1:
    print(f"\nApplying SMOTE balancing...")
    print(f"  Before: Normal={sum(y_train==0)}, Attack={sum(y_train==1)}")
    
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_selected, y_train)
    
    print(f"  After:  Normal={sum(y_train_balanced==0)}, Attack={sum(y_train_balanced==1)}")
    print(f"✓ Balanced training set: {len(X_train_balanced):,} samples")
else:
    print(f"\n⚠ Skipping SMOTE - only one class present")
    X_train_balanced, y_train_balanced = X_train_selected, y_train

## 🤖 Step 9: Train ML Models

In [ ]:
print("="*60)
print("TRAINING ML MODELS")
print("="*60)

# Random Forest
print("\n1. Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_balanced, y_train_balanced)
print("   ✓ Random Forest trained")

# XGBoost
print("\n2. Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_balanced, y_train_balanced)
print("   ✓ XGBoost trained")

# Ensemble
print("\n3. Creating Ensemble...")
ensemble_model = VotingClassifier(
    estimators=[('rf', rf_model), ('xgb', xgb_model)],
    voting='soft',
    n_jobs=-1
)
ensemble_model.fit(X_train_balanced, y_train_balanced)
print("   ✓ Ensemble created")

print("\n" + "="*60)
print("✓ ALL MODELS TRAINED")
print("="*60)

## 📊 Step 10: Comprehensive Evaluation

In [ ]:
print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# Store models
models = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'Ensemble': ensemble_model
}

# Evaluate each model
results = {}
for name, model in models.items():
    y_pred = model.predict(X_test_selected)
    y_pred_proba = model.predict_proba(X_test_selected)[:, 1]
    
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_pred_proba) if len(np.unique(y_test)) > 1 else 0
    }

# Display results
results_df = pd.DataFrame(results).T
print("\nPerformance Comparison:")
print(results_df.to_string())

# Detailed report for ensemble
y_pred_ensemble = ensemble_model.predict(X_test_selected)
print("\n" + "="*60)
print("ENSEMBLE MODEL - DETAILED METRICS")
print("="*60)
print(classification_report(y_test, y_pred_ensemble, 
                           target_names=['Normal', 'Attack'], 
                           zero_division=0))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_ensemble)
print(f"\nConfusion Matrix:")
print(f"  TN: {cm[0,0]}, FP: {cm[0,1]}")
print(f"  FN: {cm[1,0]}, TP: {cm[1,1]}")

## 📈 Step 11: Performance Visualizations

In [ ]:
# Create comprehensive visualizations
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Confusion Matrices
for idx, (name, model) in enumerate(models.items()):
    ax = fig.add_subplot(gs[0, idx])
    y_pred = model.predict(X_test_selected)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Attack'],
                yticklabels=['Normal', 'Attack'])
    ax.set_title(f'{name}\nAcc: {accuracy_score(y_test, y_pred):.4f}',
                fontsize=11, fontweight='bold')
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')

# 2. ROC Curves
ax = fig.add_subplot(gs[1, 0])
for name, model in models.items():
    y_pred_proba = model.predict_proba(X_test_selected)[:, 1]
    if len(np.unique(y_test)) > 1:
        fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
        auc = roc_auc_score(y_test, y_pred_proba)
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# 3. Precision-Recall Curves
ax = fig.add_subplot(gs[1, 1])
for name, model in models.items():
    y_pred_proba = model.predict_proba(X_test_selected)[:, 1]
    if len(np.unique(y_test)) > 1:
        precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
        ap = average_precision_score(y_test, y_pred_proba)
        ax.plot(recall, precision, label=f'{name} (AP={ap:.3f})', linewidth=2)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# 4. Feature Importance
ax = fig.add_subplot(gs[1, 2])
top_features = feature_scores.head(15)
ax.barh(range(len(top_features)), top_features['score'], color='coral')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'], fontsize=9)
ax.set_xlabel('Importance Score')
ax.set_title('Top 15 Features', fontweight='bold')
ax.invert_yaxis()

# 5. Model Comparison
ax = fig.add_subplot(gs[2, :])
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
x = np.arange(len(metrics))
width = 0.25

for i, (name, res) in enumerate(results.items()):
    values = [res[m] for m in metrics]
    ax.bar(x + i*width, values, width, label=name, alpha=0.8)
    
    # Add value labels
    for j, v in enumerate(values):
        ax.text(j + i*width, v + 0.02, f'{v:.3f}', 
               ha='center', fontsize=8)

ax.set_xlabel('Metrics', fontweight='bold')
ax.set_ylabel('Score', fontweight='bold')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1', 'ROC AUC'])
ax.legend()
ax.set_ylim([0, 1.15])
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Comprehensive Model Performance Analysis', 
            fontsize=16, fontweight='bold', y=0.995)
plt.savefig('model_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Performance visualizations saved")

## 💾 Step 12: Save Trained Models

In [ ]:
print("\n" + "="*60)
print("SAVING MODELS")
print("="*60)

# Save all models and components
model_files = {
    'mininet_ensemble_model.pkl': ensemble_model,
    'mininet_random_forest_model.pkl': rf_model,
    'mininet_xgboost_model.pkl': xgb_model,
    'mininet_scaler.pkl': scaler,
    'mininet_feature_selector.pkl': selector,
    'mininet_feature_columns.pkl': selected_features,
    'mininet_label_encoders.pkl': label_encoders
}

for filename, obj in model_files.items():
    joblib.dump(obj, filename)
    print(f"  ✓ Saved: {filename}")

# Save metadata
metadata = {
    'training_date': datetime.now().isoformat(),
    'n_samples': len(df),
    'n_normal': int(sum(df['label'] == 0)),
    'n_attack': int(sum(df['label'] == 1)),
    'n_features': len(selected_features),
    'selected_features': selected_features,
    'ensemble_metrics': {k: float(v) for k, v in results['Ensemble'].items()}
}
joblib.dump(metadata, 'mininet_model_metadata.pkl')
print(f"  ✓ Saved: mininet_model_metadata.pkl")

print("\n✓ All models saved (8 files)")

## 📥 Step 13: Download Models

In [ ]:
print("Downloading trained models...\n")

download_files = [
    'mininet_ensemble_model.pkl',
    'mininet_random_forest_model.pkl',
    'mininet_xgboost_model.pkl',
    'mininet_scaler.pkl',
    'mininet_feature_selector.pkl',
    'mininet_feature_columns.pkl',
    'mininet_label_encoders.pkl',
    'mininet_model_metadata.pkl',
    'data_analysis.png',
    'model_performance.png'
]

for file in download_files:
    if os.path.exists(file):
        files.download(file)
        print(f"  ✓ Downloaded: {file}")

print("\n✓ All files downloaded!")

## 🔮 Step 14: Make Predictions on NEW PCAP Files

In [ ]:
print("="*60)
print("PREDICTION ON NEW PCAP FILES")
print("="*60)
print("\nUpload NEW PCAP files to test predictions...\n")

new_pcaps = files.upload()
new_pcap_file = list(new_pcaps.keys())[0]

print(f"\n✓ Uploaded: {new_pcap_file}")

## 🔍 Step 15: Extract Features from New PCAP

In [ ]:
print("\nExtracting features from new PCAP...")

new_extractor = IntelligentPCAPExtractor()
new_features = new_extractor.extract_from_pcap(new_pcap_file)

new_df = pd.DataFrame(new_features)
print(f"\n✓ Extracted {len(new_df):,} flows")

# Prepare for prediction
X_new = new_df.drop(['label', 'attack_type'], axis=1, errors='ignore')

# Encode categorical
for col in X_new.select_dtypes(include=['object']).columns:
    if col in label_encoders:
        le = label_encoders[col]
        X_new[col] = X_new[col].apply(
            lambda x: le.transform([str(x)])[0] if str(x) in le.classes_ else -1
        )
    else:
        X_new[col] = LabelEncoder().fit_transform(X_new[col].astype(str))

X_new = X_new.fillna(0).replace([np.inf, -np.inf], 0)

# Align columns
for col in X.columns:
    if col not in X_new.columns:
        X_new[col] = 0
X_new = X_new[X.columns]

print("✓ Features prepared")

## 🎯 Step 16: Make Predictions

In [ ]:
print("\n" + "="*60)
print("MAKING PREDICTIONS")
print("="*60)

# Scale and select
X_new_scaled = scaler.transform(X_new)
X_new_selected = selector.transform(X_new_scaled)

# Predict
predictions = ensemble_model.predict(X_new_selected)
prediction_probas = ensemble_model.predict_proba(X_new_selected)

# Add to DataFrame
new_df['prediction'] = predictions
new_df['prediction_label'] = new_df['prediction'].map({0: 'Normal', 1: 'Attack'})
new_df['confidence'] = prediction_probas.max(axis=1)

print(f"\n✓ Predictions complete!")
print(f"\nSummary:")
print(f"  Total flows: {len(predictions):,}")
print(f"  Predicted Normal: {sum(predictions == 0):,}")
print(f"  Predicted Attack: {sum(predictions == 1):,}")
print(f"  Attack Rate: {sum(predictions == 1) / len(predictions) * 100:.1f}%")

# High-confidence attacks
high_conf = new_df[(new_df['prediction'] == 1) & (new_df['confidence'] > 0.9)]
print(f"  High-Confidence Attacks (>90%): {len(high_conf):,}")

# Display sample
print("\nSample Predictions:")
display_cols = ['protocol', 'dst_port', 'packet_count', 'packets_per_sec', 
                'prediction_label', 'confidence']
print(new_df[display_cols].head(10))

## 📊 Step 17: Prediction Analysis

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Prediction distribution
ax = axes[0]
new_df['prediction_label'].value_counts().plot(kind='bar', ax=ax, color=['green', 'red'])
ax.set_title('Prediction Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Prediction')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Confidence distribution
ax = axes[1]
new_df['confidence'].hist(bins=20, ax=ax, color='skyblue', edgecolor='black')
ax.set_title('Prediction Confidence', fontsize=14, fontweight='bold')
ax.set_xlabel('Confidence')
ax.set_ylabel('Frequency')
ax.axvline(0.9, color='red', linestyle='--', label='High Confidence')
ax.legend()

plt.tight_layout()
plt.savefig('prediction_analysis.png', dpi=300)
plt.show()

print("\n✓ Prediction analysis saved")

## 💾 Step 18: Export Predictions

In [ ]:
# Save predictions
output_file = f'predictions_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
new_df.to_csv(output_file, index=False)

print(f"✓ Predictions saved: {output_file}")
print("\nDownloading...")
files.download(output_file)
files.download('prediction_analysis.png')

print("\n" + "="*60)
print("✓ PIPELINE COMPLETE!")
print("="*60)
print("\nSummary:")
print(f"  Trained on: {len(df):,} samples")
print(f"  Model Accuracy: {results['Ensemble']['accuracy']:.4f}")
print(f"  Predicted on: {len(new_df):,} flows")
print(f"  Detected Attacks: {sum(predictions == 1):,}")
print("\n✓ All files downloaded!")
print("="*60)